In [2]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 117.5 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import pandas as pd
import streamlit as st
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler # Changed for better feature detection
from collections import Counter

# --- 1. PAGE CONFIGURATION ---
st.set_page_config(page_title="Cardio-X: Medical ECG Analysis", layout="wide")
st.title('🫀 Cardio-X: Advanced Anomaly Detection')
st.markdown("---")

# --- 2. MEDICAL KNOWLEDGE BASE ---
disease_info = {
    "Normal Beat (Class 0)": {
        "description": "The heart is beating in a healthy, regular rhythm (Sinus Rhythm).",
        "action": "Maintain a healthy lifestyle and continue regular check-ups."
    },
    "Supraventricular Ectopic Beat (Class 1)": {
        "description": "An extra heartbeat triggered by the heart's upper chambers (atria).",
        "action": "Often harmless, but if frequent, reduce caffeine/stress and consult a doctor."
    },
    "Ventricular Ectopic Beat (Class 2)": {
        "description": "An extra beat originating in the lower chambers (ventricles).",
        "action": "Frequent occurrences (PVCs) require a cardiologist's review to ensure heart health."
    },
    "Fusion Beat (Class 3)": {
        "description": "A 'collision' beat between a normal and a ventricular impulse.",
        "action": "Consult a cardiac specialist, as this indicates complex electrical conduction."
    },
    "Unclassifiable Beat (Class 4)": {
        "description": "The AI detected a high-risk or irregular pattern that doesn't fit standard categories.",
        "action": "Immediate clinical review is recommended to identify the specific arrhythmia."
    }
}

# --- 3. LOAD THE MODEL ---
@st.cache_resource
def load_ecg_model():
    model_path = r'//content/drive/MyDrive/Final_Yr/best_1d_cnn_model.keras'
    try:
        return tf.keras.models.load_model(model_path)
    except Exception as e:
        st.error(f"Critical: Model file not found at {model_path}")
        return None

model = load_ecg_model()

# --- 4. PREPROCESSING (The StandardScaler Fix) ---
def preprocess_data(df):
    try:
        # Extract the 187 numeric signal columns
        numeric_df = df.select_dtypes(include=[np.number])
        if numeric_df.shape[1] < 187:
            st.error("CSV must contain at least 187 signal columns.")
            return None

        # Take exactly 187 points
        data_array = numeric_df.iloc[:, :187].values

        # Use StandardScaler to amplify the "shape" differences of diseases
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(data_array)

        # Reshape for CNN (Samples, 187, 1)
        return X_scaled.reshape(X_scaled.shape[0], 187, 1)
    except Exception as e:
        st.error(f"Preprocessing Error: {e}")
        return None

# --- 5. MAIN INTERFACE ---
uploaded_file = st.file_uploader("Upload Test Sample to verify detection", type=["csv"])

if uploaded_file and model:
    try:
        df = pd.read_csv(uploaded_file)
        processed_data = preprocess_data(df)

        if processed_data is not None:
            # Run AI Analysis
            with st.spinner('AI analyzing complex heart patterns...'):
                predictions = model.predict(processed_data)
                predicted_indices = np.argmax(predictions, axis=1)
                confidences = np.max(predictions, axis=1) * 100

            class_labels = list(disease_info.keys())
            predicted_names = [class_labels[idx] for idx in predicted_indices]
            counts = Counter(predicted_names)

            # Dashboard Layout
            col1, col2 = st.columns([1, 1])

            with col1:
                st.write("### 📊 Analysis Summary")
                st.metric("Total Beats Analyzed", len(predicted_names))
                for label in class_labels:
                    count = counts[label]
                    icon = "✅" if label == "Normal Beat (Class 0)" else "🚨"
                    st.write(f"{icon} **{label}**: {count}")

            with col2:
                st.write("### 🔍 Row-by-Row Prediction")
                # Create a table comparing labels to AI findings
                res_df = pd.DataFrame({
                    "Actual Label": df['class_label'] if 'class_label' in df.columns else "N/A",
                    "AI Analysis": [n.split(" (")[0] for n in predicted_names],
                    "Confidence": [f"{c:.1f}%" for c in confidences]
                })
                st.dataframe(res_df, height=300)

            # --- 6. VISUALIZATION & MEDICAL INFO ---
            st.markdown("---")
            row_to_show = st.slider("Select row to visualize heartbeat", 0, len(df)-1, 0)

            v_col1, v_col2 = st.columns([2, 1])
            with v_col1:
                st.write(f"### 📈 Signal Visualization (Row {row_to_show})")
                st.line_chart(df.select_dtypes(include=[np.number]).iloc[row_to_show, :187])

            with v_col2:
                current_label = predicted_names[row_to_show]
                st.info(f"**AI Result:** {current_label}")
                st.write(f"**Description:** {disease_info[current_label]['description']}")
                st.warning(f"**Recommendation:** {disease_info[current_label]['action']}")

    except Exception as e:
        st.error(f"An error occurred: {e}")

Writing app.py


In [ ]:
import subprocess

# Install cloudflared
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!mv cloudflared-linux-amd64 /usr/local/bin/cloudflared



--2026-05-14 06:40:56--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.5.0/cloudflared-linux-amd64 [following]
--2026-05-14 06:40:56--  https://github.com/cloudflare/cloudflared/releases/download/2026.5.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/19374280-4acc-49fd-a5af-eb955320fe42?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-14T07%3A17%3A36Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-14T0

In [ ]:
# Run streamlit in the background
streamlit_process = subprocess.Popen(['streamlit', 'run', 'app.py'])

# Wait a moment for streamlit to start, then expose it via cloudflared
import time
time.sleep(5) # Give Streamlit a few seconds to start up

# Run cloudflared tunnel to expose port 8501 (default Streamlit port)
!cloudflared tunnel --url http://localhost:8501

2026-05-14T06:41:08Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-14T06:41:08Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-14T06:41:12Z INF +--------------------------------------------------------------------------------------------+
2026-05-14T06:41:12Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-14T06:41:12Z INF |  https://triangle-replace-water-grammar.trycloudflare.